In [1]:
###!/usr/bin/env python
################################################
# New style 
# ###############################################
import sys

rootdir_ = '../'
if ( rootdir_ not in sys.path ):
    sys.path.append(rootdir_)
    print( f" a path to {rootdir_} added in {__name__} ")


from Utils import GridUtils as GrU
from Utils import MakePressures as MkP
from Utils import utils as uti
from Utils import MyConstants as Co
from Utils import time_utils as tuti
from Utils import numerical_utils as nuti

import analysis_utils as auti
import file_utils as futi
import event_utils as euti
import event_io as eio


#from PyRegridding.Utils import MakePressures as MkP
#from Drivers import RegridField as RgF
import RegridField as RgF

# The usual
from datetime import date
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# for smoothing , nonlienar colors ...
from scipy.ndimage import uniform_filter
from scipy.ndimage import gaussian_filter
import matplotlib.colors as mcolors

# Some other useful packages 
import copy
import time
import cftime
import yaml
import numbers
import pickle

# Some other useful packages 
import importlib
from pathlib import Path


importlib.reload( auti )
importlib.reload( futi )
importlib.reload( euti )
importlib.reload( eio )

Rdair=Co.Rdair()


 a path to ../ added in __main__ 
 Utils.MyConstants in /glade/work/juliob/HiRes_ana_dev/Drivers/Utils 
Using Flexible parallel/serial VertRegrid 
 Utils.MyConstants in /glade/work/juliob/HiRes_ana_dev/Drivers/Utils 
 a path to /glade/work/juliob added in Utils.numerical_utils 


In [2]:
# This allow both dict.key and dict['key'] syntax
class AttrDict(dict):
    def __getattr__(self, key):
        try:
            return self[key]
        except KeyError:
            raise AttributeError(f"'AttrDict' object has no attribute '{key}'")

    def __setattr__(self, key, value):
        self[key] = value

    def __delattr__(self, key):
        try:
            del self[key]
        except KeyError:
            raise AttributeError(f"'AttrDict' object has no attribute '{key}'")



In [3]:
%%time
importlib.reload( futi )
importlib.reload( nuti )
nsteps=None
start_date=None
super_lat_range = [-90.,90.]  #[-85,-30]
#super_lat_range = [-90.,0.]  #[-85,-30]
#super_lat_range = [-80.,-30.]  #[-85,-30]
#super_lat_range = [30.,80.] # Northern Summer!!!!!!!!!!
#case, process_ncdata, start_date, nsteps = 'c153_topfix_ne240pg3_FMTHIST_xic_x02'   , False #, [2004,7,15,0], 248
#case, process_ncdata, start_date, nsteps = 'c153_topfix_ne240pg3_FMTHIST_xic_x03'   , False #, [2007,7,15,0], 248
#case, process_ncdata, start_date, nsteps = 'c153_topfix_ne240pg3_FMTHIST_xic_x03'   , False #, [2007,8,15,0], 124
case, process_ncdata  = 'cam77_dyamond1_prod1'    , False
#case, process_ncdata  = 'c124_dyamond1_prod2'    , False
#case , process_ncdata = 'xy-rdg-mm-front'    , True
#nsteps=8
A = futi.read_case( case=case, nsteps=nsteps, start_date=start_date , super_lat_range=super_lat_range ) # , nsteps = 31*8 )

time, zlev, lat, lon = A.time, A.zlev, A.lat, A.lon



  nsteps=247 
/glade/derecho/scratch/juliob/archive/cam77_dyamond1_prod1/atm/hist/DynVars_dyamond_fv1x1.2016-08-01-10800.nc /glade/derecho/scratch/juliob/archive/cam77_dyamond1_prod1/atm/hist/DynVars_dyamond_fv1x1.2016-08-31-75600.nc
 Dataset opened  , dtype of U,V float32 float32 
 Dataset opened  trimmed. -89.76 to  89.76 
Read subsetted X w open_mfdata_set  129.6591 seconds
extracted upwp,...U,V,T etc  87.0342 seconds
Vorticity calc  5.1234 seconds
rho geopht pint ... calc  35.3089 seconds
rho calc ... epwp  11.0168 seconds
(247, 58, 192, 288)
(247, 58, 192, 288)
(247, 58, 192, 288)
(247, 58, 192, 288)
tilting calc  13.7148 seconds
this is Sphere_grad_vec in numerical_utils in HiRes_ana
dlat has been adjusted to account for FUCKED UP CRAPPY fv1x1 SCRIP FILE IN CESMDATA INPUTs!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!! 
frontogenesis calc  45.4069 seconds
stability calc  4.5489 seconds
CPU times: user 1min 14s, sys: 1min 4s, total: 2min 19s
Wall time: 6min 15s


In [ ]:
A.keys()

In [ ]:
%%time

foo='/glade/derecho/scratch/juliob/archive/GW_UnitTest/xympas-TiltBoots/xympas-TiltBoots.h.2016-08-01-*.nc'
Xgw=xr.open_mfdataset( foo , data_vars='different', coords='different', compat='no_conflicts' )


ny = Xgw.sizes["ny"]
nx = Xgw.sizes["nx"]

j = np.arange(ny).repeat(nx)
i = np.tile(np.arange(nx), ny)

X2 = (
    Xgw.assign_coords(_j=("ncol", j), _i=("ncol", i))     # mapping ncol -> (j,i)
     .set_index(ncol=("_j", "_i"))
     .unstack("ncol")
     .rename({"_j": "ny", "_i": "nx"})
     .assign_coords(ny=Xgw["lat_R"], nx=Xgw["lon_R"])        # put real coords on axes
)


X2['lat']=Xgw['lat_R']
X2['lon']=Xgw['lon_R']
Xgw=X2

In [ ]:
%%time

foo='/glade/derecho/scratch/juliob/archive/GW_UnitTest/xympas-TiltBoots2/xympas-TiltBoots2.h.2016-08-01-*.nc'
Xgw2=xr.open_mfdataset( foo , data_vars='different', coords='different', compat='no_conflicts' )


ny = Xgw2.sizes["ny"]
nx = Xgw2.sizes["nx"]

j = np.arange(ny).repeat(nx)
i = np.tile(np.arange(nx), ny)

X2 = (
    Xgw2.assign_coords(_j=("ncol", j), _i=("ncol", i))     # mapping ncol -> (j,i)
     .set_index(ncol=("_j", "_i"))
     .unstack("ncol")
     .rename({"_j": "ny", "_i": "nx"})
     .assign_coords(ny=Xgw["lat_R"], nx=Xgw2["lon_R"])        # put real coords on axes
)


X2['lat']=Xgw2['lat_R']
X2['lon']=Xgw2['lon_R']
Xgw2=X2

In [ ]:
tau_mm_1 = Xgw.TAU_MOVMTN.values
tau_mm_2 = Xgw2.TAU_MOVMTN.values


In [ ]:

plt.plot( tau_mm_1[0,20,20,:] - tau_mm_2[0,20,20,:] )
#plt.plot( tau_mm_2[0,20,20,:] )



In [ ]:
%%time
k_steer  = Xgw.K_STEER_MOVMTN.values
k_launch = Xgw.K_LAUNCH_MOVMTN.values
p_steer  = Xgw.P_STEER_MOVMTN.values
p_launch = Xgw.P_LAUNCH_MOVMTN.values
pmid_mm = Xgw.PMID_MOVMTN.values
pmid    = Xgw.PMID.values

xpwp_src_1  = Xgw.XPWP_SRC_1.values
xpwp_src_3  = Xgw.XPWP_SRC_3.values





In [ ]:
%%time
tau_mm = Xgw.TAU_MOVMTN.values
pmid_mm = Xgw.PMID_MOVMTN.values


In [ ]:
lat1,lon1=Xgw.lat.values, Xgw.lon.values

z0=np.argmin( np.abs( zlev-0.))
z0p5=np.argmin( np.abs( zlev-500.))
z1=np.argmin( np.abs( zlev-1000.))
z3=np.argmin( np.abs( zlev-3000.))
z5=np.argmin( np.abs( zlev-5000.))
z6=np.argmin( np.abs( zlev-6000.))
z7=np.argmin( np.abs( zlev-7000.))
z8=np.argmin( np.abs( zlev-8000.))
z9=np.argmin( np.abs( zlev-9000.))

z10=np.argmin( np.abs( zlev-10000.))
z11=np.argmin( np.abs( zlev-11000.))
z12=np.argmin( np.abs( zlev-12000.))
z15=np.argmin( np.abs( zlev-15000.))
z17=np.argmin( np.abs( zlev-17000.))
z20=np.argmin( np.abs( zlev-20000.))
z22=np.argmin( np.abs( zlev-22000.))
z25=np.argmin( np.abs( zlev-25000.))
print( zlev[30] )

In [ ]:
#plt.contourf( xpwp_src_3[0,:,:] )
#plt.colorbar()




fig,axs=plt.subplots( 1,4, figsize=(33,6) )
ax=axs[0]
c=ax.contourf(lon1, lat1, np.mean(xpwp_src_1,axis=0) ,levels=21)
plt.colorbar(c)
ax.set_ylim(-90,90)

ax=axs[1]
c=ax.contourf(lon1, lat1, np.mean(xpwp_src_3, axis=0 ) ,levels=21)
plt.colorbar(c)
ax.set_ylim(-90,90)

ax=axs[2]
c=ax.contourf(lon1, lat1, np.mean(tau_mm[:,z25,:,:], axis=0 ) ,levels=21)
plt.colorbar(c)
ax.set_ylim(-90,90)

ax=axs[3]
c=ax.contourf(lon1, lat1, np.log10(np.mean(A.rho_epwp[:,z25,:,:], axis=0 )) ,levels=21)
plt.colorbar(c)
ax.set_ylim(-90,90)


In [ ]:
#plt.contourf( xpwp_src_3[0,:,:] )
#plt.colorbar()




fig,axs=plt.subplots( 1,4, figsize=(33,6) )
ax=axs[0]
c=ax.contourf(lon1, lat1, xpwp_src_1[0,:,:] ,levels=21)
plt.colorbar(c)
ax.set_ylim(-90,90)

ax=axs[1]
c=ax.contourf(A.lon, A.lat, np.log10(A.rho_epwp[7,z10,:,:])  ,levels=21 )
l=ax.contour(lon1, lat1, xpwp_src_1[0,:,:] ,levels=5, colors='black',alpha=0.2)
plt.colorbar(c)
ax.set_ylim(-90,90)


ax=axs[2]
c=ax.contourf(lon1, lat1, xpwp_src_3[0,:,:] ,levels=21)
plt.colorbar(c)
ax.set_ylim(-90,90)
ax=axs[3]
c=ax.contourf(A.lon, A.lat, np.log10(A.rho_epwp[7,z10,:,:])  ,levels=21 )
l=ax.contour(lon1, lat1, xpwp_src_3[0,:,:] ,levels=5, colors='black',alpha=0.2)
plt.colorbar(c)
ax.set_ylim(-90,90)


In [ ]:
##
y30s=np.argmin( np.abs( lat1 - (-30.) ))
y40s=np.argmin( np.abs( lat1 - (-40.) ))
y60s=np.argmin( np.abs( lat1 - (-60.) ))
plt.plot(  np.mean( xpwp_src_1[0, y60s:y40s,:] ,axis=(0) ) )
plt.plot(  np.mean( xpwp_src_3[0, y60s:y40s,:] ,axis=(0) ) )


In [ ]:

nt,nx,ny,nx = pmid.shape
p_steer_recon = np.zeros( (nt,ny,nx) ) 
p_launch_recon = np.zeros( (nt,ny,nx) ) 

t=0
for y in np.arange( ny ):
    for x in np.arange( nx ):
        p_steer_recon[t,y,x]  = pmid_mm[t,  int(k_steer[t,y,x]-1),   y,x]  
        p_launch_recon[t,y,x] = pmid_mm[t,  int(k_launch[t,y,x]-1),   y,x] 


In [ ]:
plt.contour( p_steer[0,:,:]-p_steer_recon[0,:,:] )

In [ ]:

#plt.plot( k_steer.flatten() )
#plt.plot( k_launch.flatten() )

plt.plot( (k_steer-k_launch).flatten() ,'.' )



In [ ]:
t,y=0,30
plt.plot( p_steer[t,y,:], '-o' )
plt.plot( p_steer_recon[t,y,:] , '-x' )

plt.xlim( 200,250)


In [ ]:
A.time[7]
zeta=Xgw.ZETA.values
tilt=Xgw.TILT.values
lat1,lon1=Xgw.lat.values, Xgw.lon.values

In [ ]:
fig,axs=plt.subplots( 1,3, figsize=(25,6) )
zetalv=0.001*np.linspace( -0.0005, 0.0005 , num=21 )
ax=axs[0]
c=ax.contourf(lon1, lat1, tilt[0,30,:,:] ,levels=zetalv )
plt.colorbar(c)
ax.set_ylim(-90,90)
ax=axs[1]
c=ax.contourf(A.lon, A.lat, A.tilt[7,30,:,:]  ,levels=zetalv )
plt.colorbar(c)
ax.set_ylim(-90,90)
ax=axs[2]
c=ax.contourf(A.lon, A.lat, np.log10(A.rho_epwp[7,30,:,:])  ,levels=21 )
plt.colorbar(c)
ax.set_ylim(-90,90)



In [ ]:
fig,axs=plt.subplots( 1,3, figsize=(25,8) )
zetalv=np.linspace( -0.0005, 0.0005 , num=21 )
ax=axs[0]
c=ax.contourf(X2.lon.values, X2.lat.values, zeta[0,30,:,:] ,levels=zetalv )
plt.colorbar(c)
ax=axs[1]
c=ax.contourf(A.lon, A.lat, A.zeta[7,30,:,:]  ,levels=zetalv )
plt.colorbar(c)
ax.set_ylim(-90,90)
ax=axs[2]
c=ax.contourf(A.lon, A.lat, A.rho_epwp[7,30,:,:]  ,levels=21 )
plt.colorbar(c)
ax.set_ylim(-90,90)



In [ ]:
X2

In [ ]:
fig,axs=plt.subplots( 1,3, figsize=(25,8) )
zetalv=np.linspace( -0.0005, 0.0005 , num=21 )
ax=axs[0]
c=ax.contourf(X2.lon.values, X2.lat.values, X2.P_STEER_MOVMVTN[0,:,:] ,levels=21 )
plt.colorbar(c)
ax.set_ylim(-90,90)
ax=axs[1]
c=ax.contourf(X2.lon.values, X2.lat.values, X2.P_LAUNCH_MOVMVTN[0,:,:] ,levels=21 )
plt.colorbar(c)
ax.set_ylim(-90,90)



In [ ]:
print( np.diff(X2.lat ) )

In [ ]:
zeta=X2.ZETA.values
zeta.shape



In [ ]:
z0=np.argmin( np.abs( zlev-0.))
z0p5=np.argmin( np.abs( zlev-500.))
z1=np.argmin( np.abs( zlev-1000.))
z3=np.argmin( np.abs( zlev-3000.))
z5=np.argmin( np.abs( zlev-5000.))
z6=np.argmin( np.abs( zlev-6000.))
z7=np.argmin( np.abs( zlev-7000.))
z8=np.argmin( np.abs( zlev-8000.))
z9=np.argmin( np.abs( zlev-9000.))

z10=np.argmin( np.abs( zlev-10000.))
z11=np.argmin( np.abs( zlev-11000.))
z12=np.argmin( np.abs( zlev-12000.))
z15=np.argmin( np.abs( zlev-15000.))
z17=np.argmin( np.abs( zlev-17000.))
z20=np.argmin( np.abs( zlev-20000.))


In [ ]:
zst,zln=z3,z8
vmag_steer  = np.sqrt( (A.u[:,zln,:,:] - A.u[:,zst,:,:])**2 + (A.v[:,zln,:,:] - A.v[:,zst,:,:] )**2 )

In [ ]:
fig,axs=plt.subplots( 1,4, figsize=(28,5) )
axs[0].contourf( np.mean( A.tilt[:,z9:z5,:,:], axis=(0,1) ) )
axs[1].contourf( np.log10(  np.mean( A.rho_epwp[:,z10,:,:], axis=(0) ) ) )
axs[2].contourf( np.mean( vmag_steer[:,:,:], axis=(0) ) ) 
axs[3].contourf( np.log10( np.mean( vmag_steer[:,:,:], axis=(0) ) * np.mean( A.tilt[:,z9:z5,:,:], axis=(0,1) ) ) ) 


In [ ]:
%%time
#################################################################
# Make event lists ... and composites
importlib.reload(euti)
importlib.reload(auti)
if recalculate == True and read_stored_El==False:
    
    #zlev_event=12_000. # before too much filtering ...
    zlev_event=10_000. # before too much filtering ...
    lat_range=  [-60,-40] #[-70,-60] #[-60,-40]
    lon_range=[0,360] # [0,60]
    exclude_orography=True
    
    fracs=[0.995,0.90,0.50,0.25,0.125,0.0625]
    #fracs=[0.95,0.90,0.50,0.25,0.125,0.0625]
    
    El = euti.make_El(
                A=A, 
                fractions_for_thresholds=fracs,
                zlev_event=zlev_event, 
                lat_range=lat_range,
                lon_range=lon_range,
                exclude_orography= exclude_orography,
                peak_footprint=(3,3),
                return_after_stage1=False
               )
    
    f=eio.pickle_write(El)
    print(f)

In [ ]:
%%time
#################################################################
# Make event lists ... and composites
importlib.reload(euti)
importlib.reload(auti)
if recalculate == True and read_stored_El==False:
    
    #zlev_event=15_000. #23_000.
    #zlev_event=12_000. # Summer
    zlev_event=10_000. # Summer
    lat_range=  [35,70] # Northern Summer!!!!!!!!!!
    lon_range=[0,360] # [0,60]
    exclude_orography=True
    
    fracs=[0.995,0.90,0.50,0.25,0.125,0.0625]
    #fracs=[0.95,0.90,0.50,0.25,0.125,0.0625]
    
    El2 = euti.make_El(
                A=A, 
                fractions_for_thresholds=fracs,
                zlev_event=zlev_event, 
                lat_range=lat_range,
                lon_range=lon_range,
                exclude_orography= exclude_orography,
                peak_footprint=(3,3),
                return_after_stage1=False
               )
    
    f=eio.pickle_write(El2)
    print(f)

In [ ]:
##
# gather your thoughts ....

poop=""
poopy = {'poop':poop}
'poop' in poopy
print( f"poopypants ? {str(poopy['poop']) } " )
print( f"poopypants ? { poopypants } " )


In [ ]:

print( np.shape(El[0].u_4D ))
print( np.shape(El[0].zlevA ))
zlev=El[0].zlevA

In [ ]:
#import random_forest as RF

Eco=   El[0]  #euti.combine_event_dicts(El[3], El[1], label_key='event_strength')

z0=np.argmin( np.abs( zlev-0.))
z3=np.argmin( np.abs( zlev-3000.))
z5=np.argmin( np.abs( zlev-5000.))
z6=np.argmin( np.abs( zlev-6000.))
z7=np.argmin( np.abs( zlev-7000.))
z10=np.argmin( np.abs( zlev-10000.))
z11=np.argmin( np.abs( zlev-11000.))
z12=np.argmin( np.abs( zlev-12000.))
z15=np.argmin( np.abs( zlev-15000.))


In [ ]:
print( np.shape(Eco.u_4D ))
dt = Eco.timeA[1] - Eco.timeA[0] 
print(dt.total_seconds())

In [ ]:
import importlib                                                                                                                                                                                                                                                                                              
import mlp_utils as mlu  
import predictors as Predi
importlib.reload(mlu) 
#importlib.reload(RF)
importlib.reload(Predi)
importlib.reload(eio)


#zlev=Eco.zlev.values

nv,nt_v,nz_v,ny_v,nx_v = np.shape( Eco.zeta_4D )

use_predictors= ['tilt_4D', 'zeta_4D', 'th_4D', 'u_4D','v_4D']
use_predictors= ['tilt_4D', 'zeta_4D', 'th_4D'] #, 'u_4D','v_4D']
use_predictors= ['tilt_4D', 'zeta_4D', 'stab_4D'] #, 'u_4D','v_4D']
#use_predictors= ['fgf_4D', 'zeta_4D']
trange = [nt_v-1, nt_v ] # No memory"
yrange = [1,-1] #None #[5,ny_v]
use_MM_winds=True

#Use Defaults ... from a while ago
use_predictors= None 
trange = None 
yrange = None 
use_MM_winds=False


#========================================================================================================================================================
#-----------------------------------
# Reasonbly succesful predictor set for
# /glade/derecho/scratch/juliob/archive/GW_event_analysis/PKL/cam77_dyamond1_prod1_2016-08-01-10800-x-2016-08-31-75600_60S-40S_ocean_EvZ10km_rho_epwp.pkl
# target was rho_epwp at z=10km 
#-----------------------------------
#use_predictors= ['tilt_4D', 'zeta_4D', 'stab_4D', 'u_4D','v_4D']
use_predictors= ['tilt_4D', 'zeta_4D', 'stab_4D']
yrange = [1,-1]
use_MM_winds=True

predictors,predictor_names,use_predictors,key_z,short_desc = Predi.set_A_MM_genl( Eco=Eco, zlev=zlev, 
                                                                          trange=trange,
                                                                          yrange=yrange,
                                                                          use_predictors= use_predictors, 
                                                                          use_MM_winds=use_MM_winds)
targ_scaling=1.
z_targ=z10
yv=targ_scaling*np.mean( Eco.epwp_4D[:,-1,z_targ,:,:],axis=(2,1) )

#========================================================================================================================================================

#predictors,predictor_names,use_predictors,key_z = predictors.small_MM_set( Eco=Eco, zlev=zlev )



print( f"Predictors = {use_predictors}" )
print( f"key_z levels (m) = {[int(zlev[z]) for z in key_z]}" )
print( f"n_predictors = {len(predictors)}" )
print(f"target scaled by {targ_scaling}")
print( f"Events in {Eco.case},latlon={Eco.lon_range}X{Eco.lat_range}, exclude orography={Eco.exclude_orography} " )
print( f"Peak footprint={Eco.peak_footprint}" )
#print( f"Dates {A.start_date} to {A.end_date}, nsteps={A.nsteps}, stepsize={A.step_size} hrs" )
print( '\n' )

mlp_model, mlp_results = mlu.fit_mlp_general(
    predictors       = predictors,        # same list already built above
    predictor_names  = predictor_names,
    target           = yv,
    event_times      = Eco.time4D,
    train_interval   = (48, 248),         # same split as the RF
    test_interval    = (0, 28),
    dropout=0.1,
    hidden_dims      = (256, 256, 256, 128),
    weight_decay     = 1e-5,
    patience         = 100,
    lr_patience      = 3,
    batch_size=512,
    loss_power = 5,
    log_predictor_patterns = ['tilt','U_wv_src_mm',],)   # <-- new )

target_ = f"Target: {Eco.fld} at {zlev[z_targ]/1000.:.0f}km"
short_desc=f"{target_}: {short_desc}"

mlu.plot_distributions(mlp_results, desc=short_desc  ) 

mlu.plot_mlp_results(mlp_results, top_n=20 ,desc=short_desc[0:60])

In [ ]:
print( type(Eco) == list )
print(f"Target: {Eco.fld} at {zlev[z_targ]/1000.:.0f}km")

In [ ]:

meta = {
    'feat_scaler':   mlp_results['feat_scaler'],
    'target_scaler': mlp_results['target_scaler'],
    'log_pred_mask': mlp_results['log_pred_mask'],
}
device = 'cpu'  # or 'cuda' if you're on GPU

In [ ]:
y_pred_new = mlu.apply_mlp(mlp_model, meta, device, predictors)

In [ ]:
from scipy import stats

print(y_pred_new.shape)
print(yv.shape)



eps = 1e-12
ly_pred_new       = np.log(np.maximum(y_pred_new,  eps))
ly_targ           = np.log(np.maximum(yv        ,  eps))
r_log, _ = stats.pearsonr(ly_targ, ly_pred_new)
print(r_log )

In [ ]:
######### TRANSFER LEARNING ????????  #################################################


Eco2 = El2[0]



nv,nt_v,nz_v,ny_v,nx_v = np.shape( Eco2.zeta_4D )

"""
# SHould NOT reset these from above!!!!!!!!!!!!!!!!
#Use Defaults ... from a while ago
use_predictors= None 
trange = None 
yrange = None 
use_MM_winds=False

use_predictors= ['tilt_4D', 'zeta_4D', 'stab_4D', 'u_4D','v_4D']
yrange = [1,-1]



use_predictors= ['tilt_4D', 'zeta_4D', 'stab_4D']
yrange = [1,-1]
use_MM_winds=True
"""


predictors_2,predictor_names,use_predictors,key_z,short_desc = Predi.set_A_MM_genl( Eco=Eco2, zlev=zlev, 
                                                                          trange=trange,
                                                                          yrange=yrange,
                                                                          use_predictors= use_predictors, 
                                                                          use_MM_winds=use_MM_winds)

targ_scaling=1.
yv_2=targ_scaling*np.mean( Eco2.epwp_4D[:,-1,z_targ,:,:],axis=(2,1) )

target_ = f"Target: {Eco2.fld} at {zlev[z_targ]/1000.:.0f}km"
short_desc=f"{target_}: {short_desc}"


In [ ]:
y_pred_2 = mlu.apply_mlp(mlp_model, meta, device, predictors_2)

In [ ]:
from scipy import stats

print(y_pred_2.shape)
print(yv_2.shape)



eps = 1e-12
ly_pred_2       = np.log(np.maximum(y_pred_2,  eps))
ly_targ_2       = np.log(np.maximum(yv_2        ,  eps))
r_log, _ = stats.pearsonr(ly_targ_2, ly_pred_2)
print(r_log )

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(6, 5))

# --- panel 1: log-log scatter ---
ax = axes
ax.scatter(ly_targ_2, ly_pred_2, alpha=0.3, s=10, color='steelblue')
lims = [min(ly_targ_2.min(), ly_pred_2.min()),
        max(ly_targ_2.max(), ly_pred_2.max())]
ax.plot(lims, lims, 'r--', lw=1)
r_log, _ = stats.pearsonr(ly_targ_2, ly_pred_2)
ax.set_xlabel('log(actual)')
ax.set_ylabel('log(predicted)')
ax.set_title(f'Log-space scatter  r={r_log:.3f}')



In [ ]:
print( short_desc[0:60] )
print( short_desc)


In [ ]:
"""


# fit
rf, results = RF.fit_rf_general(predictors=predictors, 
                                   predictor_names=predictor_names, 
                                   target=yv,
                                   event_times = Eco.time4D ,
                                   train_interval   = (48,248),
                                   test_interval    = (0,28),
                                   min_samples_leaf=10,
                                    )

RF.plot_rf_results(results, top_n=20)
"""

In [ ]:
print(np.diff(Eco.u_4D.shape))

In [ ]:
%%time
#importlib.reload(eio)
f=eio.pickle_write(El)
print(f)

In [ ]:
print(A.epwp.shape)
print( A.epwp.shape[0] * A.epwp.shape[2] * A.epwp.shape[3] )


In [ ]:
importlib.reload( auti )
auti.plot_xavg_compos( fld='zeta_4D', El=El )
auti.plot_xavg_compos( fld='tilt_4D', El=El )
auti.plot_xavg_compos( fld='fgf_4D', El=El )


In [ ]:
importlib.reload( auti )
auti.plot_xavg_compos( fld='zeta_4D', El=El2 )
auti.plot_xavg_compos( fld='tilt_4D', El=El2 )
auti.plot_xavg_compos( fld='fgf_4D', El=El2 )


In [ ]:

ds=El[0].ds
print(int(ds.itime.values.max()) + 1)

ds_drop=ds.drop_vars( ['lat','lon','zlev'] )


event_list=euti.ds_to_event_list( ds_drop )

In [ ]:
evoo=event_list[120]
len(evoo)
np.array(evoo[1]['ix'])

In [ ]:
Eco.zeta_4D.shape

In [ ]:
#### SAVE OFF events dataset
_t=0
euti.write_ds(ds=El[_t].ds, A=A,fraction_of_total=El[_t].Frac_of_total_epwp,
            zlev_event=zlev_event,thresh=El[_t].threshold,
            lat_range=lat_range,lon_range=lon_range,extra_info='_NoTopo' )


In [ ]:
from scipy import stats

#zlev=Eco.zlev

print(Eco.epwp_4D.shape)

#xv=Eco.zeta_4D[:,1,z6,4,:].mean(axis=1)
#xv=np.mean(Eco.zeta_4D[:,:,z10,4,:],axis=(1,2))
#xv=np.mean(Eco.zeta_4D[:,:,z7,:,:],axis=(1,2,3))
#xv=np.mean( Eco.tilt_4D[:,:,z7,:,:],axis=(1,2,3)  ) 

xvs=[]
xvs.append( np.mean( Eco.zeta_4D[:,:,z10:z3,:,:],axis=(1,2,3,4)  ) )
xvs.append( np.mean( Eco.tilt_4D[:,:,z10:z3,:,:],axis=(1,2,3,4)  ) )
xvs.append( np.mean( Eco.fgf_4D[:,:,z10:z3,:,:],axis=(1,2,3,4)  ) )
flds=['zeta','tilt','fgf']

yv=np.mean( Eco.epwp_4D[:,:,z10,:,:],axis=(3,2,1) )
#yv=np.mean( Eco.epwp_4D[:,nt_v-1,z10,:,:],axis=(2,1) )
print(yv.shape)

fig,axs=plt.subplots( 1, len(xvs), figsize=( (len(xvs)*8, 4 ) ) )
ip=0
for xv in xvs:
    ax=axs[ip]
    ax.scatter( xv,yv )
    r, p = stats.pearsonr(xv, yv)
    print(f"Patch mean {flds[ip]} vs patch mean epwp: r={r:.3f}, p={p:.2e}")
    ip=ip+1


In [ ]:
evoo=event_list[120]
fig,ax=plt.subplots( figsize=(20,8) )
ax.contourf(lon,lat,np.log(A.rho_epwp[120,z12,:,:]), levels=51, cmap='coolwarm')
for ev in evoo:
    ax.scatter( lon[ev['ix']] , lat[ev['iy']], marker='+' , c='black')
ax.contour( lon, lat, A.htopo , levels=[0.1,1,10,100,200,1000] )

In [ ]:
Eco = euti.combine_event_dicts(El[0], El2[0], label_key='event_strength')


In [ ]:
Eco['case']='Combined NH/SH'
Eco['exclude_orography']= True #'Combined NH/SH'


In [ ]:
#del Eco
Eco=El[0]
Eco.stab_4D=10_000.*Eco.stab_4D


In [ ]:

import importlib                                                                                                                                                                                                                                                                                              
import mlp_utils as mlu  
import predictors
importlib.reload(mlu) 
importlib.reload(RF)
importlib.reload(predictors)


#zlev=Eco.zlev.values

nv,nt_v,nz_v,ny_v,nx_v = np.shape( Eco.zeta_4D )

use_predictors= ['tilt_4D', 'zeta_4D', 'th_4D', 'u_4D','v_4D']
use_predictors= ['tilt_4D', 'zeta_4D', 'th_4D'] #, 'u_4D','v_4D']
use_predictors= ['tilt_4D', 'zeta_4D', 'stab_4D'] #, 'u_4D','v_4D']
#use_predictors= ['fgf_4D', 'zeta_4D']
trange = [nt_v-1, nt_v ] # No memory"
yrange = [1,-1] #None #[5,ny_v]
use_MM_winds=True

predictors,predictor_names,use_predictors,key_z = predictors.set_A_MM_genl( Eco=Eco, zlev=zlev, 
                                                                          trange=trange,
                                                                          yrange=yrange,
                                                                          use_predictors= use_predictors, 
                                                                          use_MM_winds=use_MM_winds)
#predictors,predictor_names,use_predictors,key_z = predictors.small_MM_set( Eco=Eco, zlev=zlev )


targ_scaling=1.
yv=targ_scaling*np.mean( Eco.epwp_4D[:,-1,z12,:,:],axis=(2,1) )

print( f"Predictors = {use_predictors}" )
print( f"key_z levels (m) = {[int(zlev[z]) for z in key_z]}" )
print( f"n_predictors = {len(predictors)}" )
print(f"target scaled by {targ_scaling}")
print( f"Events in {Eco.case},latlon={Eco.lon_range}X{Eco.lat_range}, exclude orography={Eco.exclude_orography} " )
print( f"Peak footprint={Eco.peak_footprint}" )
#print( f"Dates {A.start_date} to {A.end_date}, nsteps={A.nsteps}, stepsize={A.step_size} hrs" )
print( '\n' )

mlp_model, mlp_results = mlu.fit_mlp_general(
    predictors       = predictors,        # same list already built above
    predictor_names  = predictor_names,
    target           = yv,
    event_times      = Eco.time4D,
    train_interval   = (48, 248),         # same split as the RF
    test_interval    = (0, 28),
    dropout=0.1,
    hidden_dims      = (256, 256, 256, 128),
    weight_decay     = 1e-5,
    patience         = 100,
    lr_patience      = 3,
    batch_size=512,
    loss_power = 5,
    log_predictor_patterns = ['tilt','U_wv_src_mm',],)   # <-- new )

mlu.plot_distributions(mlp_results)   # <-- the new diagnostic  




"""


# fit
rf, results = RF.fit_rf_general(predictors=predictors, 
                                   predictor_names=predictor_names, 
                                   target=yv,
                                   event_times = Eco.time4D ,
                                   train_interval   = (48,248),
                                   test_interval    = (0,28),
                                   min_samples_leaf=10,
                                    )

RF.plot_rf_results(results, top_n=20)
"""


In [ ]:
print( f"{predictor_names}")




In [ ]:
mlu.plot_mlp_results(mlp_results, top_n=20)

In [ ]:
print( El[0].keys() )

In [ ]:
print(poopypants)

In [ ]:
importlib.reload( mlu )

mlu.save_mlp(
    model        = mlp_model,
    results      = mlp_results,
    path_stem    = f'gw_mlp_{Eco.case}',   # e.g. 'gw_mlp_dyamond' — one file per case
    # architecture
    hidden_dims  = (256, 256, 256, 128),
    dropout      = 0.1,
    # training hyperparameters
    loss_power          = 5,
    log_predictor_eps   = 1e-30,           # default — change if you overrode it
    lr                  = 1e-3,            # default — change if you overrode it
    weight_decay        = 1e-5,
    batch_size          = 512,
    random_state        = 42,              # default
)

In [ ]:
importlib.reload( mlu )

# --- load ---
mlp_model_loaded, meta, device = mlu.load_mlp(f'gw_mlp_{Eco.case}')

# --- run both on the same predictors ---
y_pred_orig   = mlu.apply_mlp(mlp_model,        meta, device, predictors)
y_pred_loaded = mlu.apply_mlp(mlp_model_loaded,  meta, device, predictors)

In [ ]:
# --- compare ---
max_diff = np.max(np.abs(y_pred_orig - y_pred_loaded))
print(f"Max absolute difference: {max_diff:.2e}")
assert np.allclose(y_pred_orig, y_pred_loaded, atol=1e-6), \
    "Mismatch exceeds 1e-6 — something went wrong in save/load"
print("OK — loaded model is bit-identical to original")

In [ ]:
import pickle

path_stem=f'/glade/derecho/scratch/juliob/El_save_test'
with open(f'{path_stem}.pkl', 'wb') as f:
    pickle.dump(El, f)


In [ ]:
fig,axs=plt.subplots(1,2,figsize=(15,8) )

ax=axs[0]
co=ax.contourf(A.lon,A.lat,np.log(np.mean( A.epwp[:,z15,:,:],axis=0)) )
#co=ax.contourf(A.lon,A.lat,np.log10(np.mean( A.tilt[:,z3,:,:],axis=0)+1.e-8) )
ax.contour(A.lon,A.lat, A.htopo )
plt.colorbar(co)

ax=axs[1]
#plt.contourf(A.lon,A.lat,np.log(np.mean( A.epwp[:,z15,:,:],axis=0)) )
co=ax.contourf(A.lon,A.lat,np.log10(np.mean( A.tilt[:,z10:z3,:,:],axis=(0,1) )+1.e-8) ,levels=21 )
ax.contour(A.lon,A.lat, A.htopo )
plt.colorbar(co)


In [ ]:
Pi  = Co.pi()
rlat = (Pi/180.)*lat
coslat = np.cos( rlat )

coslat_jn = np.roll(coslat, -1)
coslat_js = np.roll(coslat, 1)


coslat_jn[-1] = coslat[-1]  # repeat last row
coslat_js[0] = coslat[0]  # repeat last row


plt.plot( coslat )
plt.plot( coslat_jn )
plt.plot( coslat_js )



In [ ]:
boo=np.log10(np.mean( A.tilt[:,z10:z3,:,:],axis=(0,1) )+1.e-8) 
boo=np.mean( A.u[:,z7:z7+1,:,:],axis=(0,1)  )
#plt.plot( boo[:,30] )
plt.plot( A.lat )
